In [2]:
from pathlib import Path

base = "C:/Users/colin/projects/UW/Project/LA/LA"
train_files = base + "/ASVspoof2019_LA_train/flac"
dev_files = base + "/ASVspoof2019_LA_dev/flac"
eval_files = base + "/ASVspoof2019_LA_eval/flac"
train_protocols = base + "/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
dev_protocols = base + "/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt"
eval_protocols = base + "/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt"

In [3]:
train_audio_files = list(Path(train_files).glob("*.flac"))
dev_audio_files = list(Path(dev_files).glob("*.flac"))
eval_audio_files = list(Path(eval_files).glob("*.flac"))

In [4]:
import pandas as pd

train_df = pd.read_csv(train_protocols, sep=r"\s+", header=None)
dev_df = pd.read_csv(dev_protocols, sep=r"\s+", header=None)
eval_df = pd.read_csv(eval_protocols, sep=r"\s+", header=None)

In [4]:
label_map = {
    **dict(zip(train_df[1], train_df[4])),
    **dict(zip(dev_df[1], dev_df[4])),
    **dict(zip(eval_df[1], eval_df[4]))
}

In [5]:
import librosa
import numpy as np

def extract_features(audio_path, sr=16000, n_mfcc=13):
    # Load audio
    audio, sr = librosa.load(audio_path, sr=sr)

    # MFCCs
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
    mfcc_mean = mfcc.mean(axis=1)
    mfcc_std = mfcc.std(axis=1)

    mfcc_features = np.concatenate([mfcc_mean, mfcc_std])

    # Spectral features
    spec_centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)
    spec_rolloff = librosa.feature.spectral_rolloff(y=audio, sr=sr)
    zcr = librosa.feature.zero_crossing_rate(audio)
    
    spectral_features = np.array([
        spec_centroid.mean(),
        spec_centroid.std(),
        spec_rolloff.mean(),
        spec_rolloff.std(),
        zcr.mean(),
        zcr.std()
    ])

    # Combine features
    feature_vector = np.concatenate([mfcc_features, spectral_features])

    return feature_vector

In [6]:
from tqdm import tqdm

def establish_dataset(audio_files):
    X = []
    y = []

    for audio_path in tqdm(audio_files):
        audio_id = audio_path.stem
        if audio_id not in label_map:
            continue
        X.append(extract_features(audio_path))
        y.append(0 if label_map[audio_id] == "bonafide" else 1) # Labels are either "bonafide" or "spoof"

    return (np.array(X), np.array(y))

In [7]:
X_train, y_train = establish_dataset(train_audio_files)

100%|████████████████████████████████████████████████████████████████████████████| 25380/25380 [13:11<00:00, 32.08it/s]


In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
y_train = y_train

In [9]:
from sklearn.svm import SVC

model = SVC(kernel='rbf', C=1.0, gamma='auto')
model.fit(X_train, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'auto'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [10]:
X_dev, y_dev = establish_dataset(dev_audio_files)
X_eval, y_eval = establish_dataset(eval_audio_files)

# Re-use scaler from training and just transform
X_dev = scaler.transform(X_dev)
X_eval = scaler.transform(X_eval)

100%|████████████████████████████████████████████████████████████████████████████| 71933/71933 [29:46<00:00, 40.26it/s]


In [15]:
y_dev_pred = model.predict(X_dev)
y_eval_pred = model.predict(X_eval)

In [18]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

accuracy = accuracy_score(y_dev, y_dev_pred)
print(f"Model Accuracy (against dev): {accuracy:.2f}")
confusion = confusion_matrix(y_dev, y_dev_pred)
print(confusion)

accuracy = accuracy_score(y_eval, y_eval_pred)
print(f"Model Accuracy (against eval): {accuracy:.2f}")
confusion = confusion_matrix(y_eval, y_eval_pred)
print(confusion)

print("\n----------\n")

classification = classification_report(y_dev, y_dev_pred)
print("Classification Report (against dev):")
print(classification)
classification = classification_report(y_eval, y_eval_pred)
print("Classification Report (against eval):")
print(classification)

Model Accuracy (against dev): 0.95
[[ 1512  1036]
 [  285 22011]]
Model Accuracy (against eval): 0.90
[[ 5005  2350]
 [ 4951 58931]]

----------

Classification Report (against dev):
              precision    recall  f1-score   support

           0       0.84      0.59      0.70      2548
           1       0.96      0.99      0.97     22296

    accuracy                           0.95     24844
   macro avg       0.90      0.79      0.83     24844
weighted avg       0.94      0.95      0.94     24844

Classification Report (against eval):
              precision    recall  f1-score   support

           0       0.50      0.68      0.58      7355
           1       0.96      0.92      0.94     63882

    accuracy                           0.90     71237
   macro avg       0.73      0.80      0.76     71237
weighted avg       0.91      0.90      0.90     71237



In [17]:
from sklearn.metrics import roc_curve, roc_auc_score

def compute_eer(y_true, y_scores):
    fpr, tpr, thresholds = roc_curve(y_true, y_scores, pos_label=1)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2.0
    return eer

y_dev_scores = model.decision_function(X_dev)
y_eval_scores = model.decision_function(X_eval)

auc_score = roc_auc_score(y_dev, y_dev_scores)
print(f"Model AUC (against dev): {auc_score:.2f}")
err_score = compute_eer(y_dev, y_dev_scores)
print(f"Model ERR (against dev): {err_score:.2f}")

auc_score = roc_auc_score(y_eval, y_eval_scores)
print(f"Model AUC (against eval): {auc_score:.2f}")
err_score = compute_eer(y_eval, y_eval_scores)
print(f"Model ERR (against eval): {err_score:.2f}")

Model AUC (against dev): 0.96
Model ERR (against dev): 0.10
Model AUC (against eval): 0.92
Model ERR (against eval): 0.15
